# Reading ROOT files with uproot

<br><br><br>

## Introduction

[uproot](https://uproot.readthedocs.io/) is a pure-Python library for reading and writing ROOT files. It does **not** depend on ROOT itself — it speaks the ROOT binary format directly, returning data as NumPy arrays or Awkward Arrays.

**RNTuple** is the current default storage format for new CMS data. It replaces the older TTree format and is better designed for modern storage systems, parallel I/O, and the columnar workflows you learned in the previous tutorial. uproot has fully supported reading RNTuples since v5.3 and writing since v5.7 (where it became the default).

Key concepts:

* A ROOT file is a **key–value store**: keys are strings, values are ROOT objects (histograms, RNTuples, TTrees, ...).
* An **RNTuple** is a table: **fields** are columns, **entries** are rows (usually events). (TTree called them "branches"; the RNTuple equivalent is "fields".)
* uproot reads data **lazily** — nothing is loaded from disk until you ask for it.
* The output is an **Awkward Array**, which plugs straight into everything you learned in the columnar-analysis tutorial.

<br><br><br>

## Opening a file

`uproot.open()` works the same way for RNTuple files as for TTree files. The `with` statement is recommended so the file handle is always closed cleanly.

In [1]:
import uproot
import skhep_testdata

In [13]:
path = skhep_testdata.data_path("Run2012BC_DoubleMuParked_Muons_1000evts_rntuple_v1-0-0-0.root")

In [15]:
with uproot.open(path) as file:
    print(file["Events"].keys())

['_collection0', '_collection0.Muon_pt', '_collection0.Muon_eta', '_collection0.Muon_phi', '_collection0.Muon_mass', '_collection0.Muon_charge', 'Muon_pt', 'Muon_eta', 'Muon_phi', 'Muon_mass', 'Muon_charge', 'nMuon']


<br><br><br>

`file` behaves like a Python `dict`. `.keys()` lists everything stored in the file, and `.classnames()` shows each object's type.

In [16]:
with uproot.open(path) as file:
    print(file.classnames())

{'Events;1': 'ROOT::RNTuple'}


<br><br><br>

You can jump straight to an object by putting its name after a colon in the path string:

In [17]:
# These two are equivalent:
# uproot.open(path)["Events"]
# uproot.open("data/SMHiggsToZZTo4L.root:Events")
ntuple = uproot.open(path)["Events"] #uproot.open("data/SMHiggsToZZTo4L.root:Events")
ntuple

<ROOT::RNTuple 'Events' (7 fields) at 0x000138336850>

<br><br><br>

## Inspecting an RNTuple

RNTuple uses the word **field** where TTree used **branch**. `.keys()` lists field names; `.show()` includes their types.

In [22]:
with uproot.open(path) as ntuple:
    ntuple["Events"].show()

Events (ROOT::RNTuple)
├─ _collection0 ()
│  ├─ Muon_pt (float)
│  ├─ Muon_eta (float)
│  ├─ Muon_phi (float)
│  ├─ Muon_mass (float)
│  └─ Muon_charge (std::int32_t)
├─ Muon_pt (ROOT::VecOps::RVec<float>)
├─ Muon_eta (ROOT::VecOps::RVec<float>)
├─ Muon_phi (ROOT::VecOps::RVec<float>)
├─ Muon_mass (ROOT::VecOps::RVec<float>)
├─ Muon_charge (ROOT::VecOps::RVec<std::int32_t>)
└─ nMuon (ROOT::RNTupleCardinality<std::uint32_t>)


<br><br><br>

Filter by glob pattern — works the same as for TTrees:

In [25]:
with uproot.open(path) as ntuple:
    print(ntuple["Events"].keys(filter_name="Muon_*"))

['_collection0.Muon_pt', '_collection0.Muon_eta', '_collection0.Muon_phi', '_collection0.Muon_mass', '_collection0.Muon_charge', 'Muon_pt', 'Muon_eta', 'Muon_phi', 'Muon_mass', 'Muon_charge']


<br><br><br>

Check the entry count:

In [27]:
with uproot.open(path) as ntuple:
    print(f"{ntuple["Events"].num_entries:,} events")

1,000 events


<br><br><br>

## Reading arrays

### Reading a single field

`.array()` reads one field. Variable-length fields (particles per event) come back as ragged Awkward Arrays; flat fields (one value per event) come back as NumPy arrays.

In [28]:
with uproot.open(path) as ntuple:
    muon_pt = ntuple["Events"]["Muon_pt"].array()

muon_pt

<Array [[10.8, 15.7], ..., [28.9, 8.62, 4.51]] type='1000 * var * float32'>

<br><br><br>

In [29]:
with uproot.open(path) as ntuple:
    n_muon = ntuple["Events"]["nMuon"].array()

print(type(n_muon))
n_muon

<class 'awkward.highlevel.Array'>


<Array [2, 2, 1, 4, 4, 3, 2, 2, ..., 2, 1, 3, 2, 3, 4, 3] type='1000 * int64'>

<br><br><br>

### Reading multiple fields at once

`.arrays()` (plural) reads a set of fields and returns an Awkward record array — the same structure the columnar tutorial started with.

In [30]:
with uproot.open(path) as ntuple:
    muons = ntuple["Events"].arrays(filter_name="Muon_*")

muons

<Array [{_collection0: [...], ...}, ..., {...}] type='1000 * {_collection0:...'>

<br><br><br>

In [31]:
muons["Muon_pt"]

<Array [[10.8, 15.7], ..., [28.9, 8.62, 4.51]] type='1000 * var * float32'>

<br><br><br>

### Reading a slice of entries

Use `entry_start` and `entry_stop` for quick checks on large files:

In [32]:
with uproot.open(path) as ntuple:
    first_thousand = ntuple["Events"].arrays(filter_name="Muon_*", entry_start=0, entry_stop=1000)

print(f"{len(first_thousand):,} events read")

1,000 events read


<br><br><br>

## Inspecting a single event

Indexing into the record array gives you one event as an Awkward `Record`. `.tolist()` converts it to plain Python dicts and lists:

In [33]:
with uproot.open(path) as ntuple:
    dataset = ntuple["Events"].arrays()

dataset[1].tolist()  # second event

{'_collection0': [{'Muon_pt': 10.538490295410156,
   'Muon_eta': -0.42778006196022034,
   'Muon_phi': -0.2747921049594879,
   'Muon_mass': 0.10565836727619171,
   'Muon_charge': 1},
  {'Muon_pt': 16.327096939086914,
   'Muon_eta': 0.34922507405281067,
   'Muon_phi': 2.539781332015991,
   'Muon_mass': 0.10565836727619171,
   'Muon_charge': -1}],
 'Muon_pt': [10.538490295410156, 16.327096939086914],
 'Muon_eta': [-0.42778006196022034, 0.34922507405281067],
 'Muon_phi': [-0.2747921049594879, 2.539781332015991],
 'Muon_mass': [0.10565836727619171, 0.10565836727619171],
 'Muon_charge': [1, -1],
 'nMuon': 2}

<br><br><br>

## Choosing which fields to read

Reading only what you need is important for performance with large files.

<br><br><br>

**Explicit list:**

In [34]:
with uproot.open(path) as ntuple:
    muon_kin = ntuple["Events"].arrays(["Muon_pt", "Muon_eta", "Muon_phi", "Muon_mass", "Muon_charge"])

muon_kin

<Array [{_collection0: [...], ...}, ..., {...}] type='1000 * {_collection0:...'>

<br><br><br>

**Glob pattern** — wildcard matching on field names:

In [35]:
with uproot.open("data/SMHiggsToZZTo4L.root:Events") as ntuple:
    electron_fields = ntuple.arrays(filter_name="Electron_*")

print(electron_fields.fields)

['Electron_pt', 'Electron_eta', 'Electron_phi', 'Electron_mass', 'Electron_charge', 'Electron_pfRelIso03_all', 'Electron_dxy', 'Electron_dxyErr', 'Electron_dz', 'Electron_dzErr']


<br><br><br>

**Regular expression** — for precise multi-pattern selection:

In [ ]:
import re

with uproot.open("data/SMHiggsToZZTo4L.root:Events") as ntuple:
    # all _pt and _eta fields for any particle type
    kin = ntuple.arrays(filter_name=re.compile(r".+_(pt|eta)$"))

print(kin.fields)

<br><br><br>

## Output formats

The `library` argument controls what uproot returns. The default is `"ak"` (Awkward Array), which handles ragged data naturally.

### `library="ak"` (default)

In [ ]:
with uproot.open("data/SMHiggsToZZTo4L.root:Events") as ntuple:
    ak_out = ntuple["Muon_pt"].array(library="ak")

print(type(ak_out))
ak_out

<br><br><br>

### `library="np"` — NumPy (flat fields only)

In [ ]:
with uproot.open("data/SMHiggsToZZTo4L.root:Events") as ntuple:
    np_out = ntuple["nMuon"].array(library="np")

print(type(np_out))
np_out[:10]

<br><br><br>

### `library="pd"` — pandas DataFrame

Only works for flat (non-ragged) fields. Use Awkward Array for variable-length fields like `Muon_pt`.

In [ ]:
with uproot.open("data/SMHiggsToZZTo4L.root:Events") as ntuple:
    df = ntuple.arrays(["nMuon", "nElectron", "MET_pt", "MET_phi"], library="pd")

df.head(10)

<br><br><br>

## Processing files too large to fit in memory

`uproot.iterate()` processes a file (or many files) in chunks, never loading the whole dataset at once. It works identically for RNTuples and TTrees.

In [ ]:
import awkward as ak

total_muon_pt_sum = 0
total_events = 0

for batch in uproot.iterate(
    "data/SMHiggsToZZTo4L.root:Events",
    expressions=["Muon_pt"],
    step_size=10_000,           # entries per chunk
):
    total_muon_pt_sum += ak.sum(ak.flatten(batch["Muon_pt"]))
    total_events += len(batch)

print(f"Processed {total_events:,} events")
print(f"Sum of all muon pT values: {total_muon_pt_sum:.1f} GeV")

<br><br><br>

Globs and lists of files work too — uproot crosses file boundaries automatically:

In [ ]:
# Example — adapt the glob to your own dataset:
#
# for batch in uproot.iterate(
#     "data/Run3_*.root:Events",          # glob over many files
#     expressions=["Muon_pt", "Muon_eta"],
#     step_size="100 MB",                 # memory-based chunk size
# ):
#     process(batch)
print("(pattern shown — adapt to your own dataset)")

<br><br><br>

### `uproot.concatenate()` — all at once

When the dataset fits in memory, `concatenate` is simpler than iterating:

In [ ]:
dataset = uproot.concatenate(
    "data/SMHiggsToZZTo4L.root:Events",
    filter_name="Muon_*",
)
dataset

<br><br><br>

## Connecting back to the columnar tutorial

Once you have an Awkward Array from uproot, everything from the columnar tutorial applies. Here is the full pipeline: file → Lorentz vectors → dimuon mass spectrum.

In [ ]:
import awkward as ak
import vector
vector.register_awkward()

In [ ]:
with uproot.open("data/SMHiggsToZZTo4L.root:Events") as ntuple:
    raw = ntuple.arrays(["Muon_pt", "Muon_eta", "Muon_phi", "Muon_mass", "Muon_charge"])

muons = ak.zip({
    "pt":     raw["Muon_pt"],
    "eta":    raw["Muon_eta"],
    "phi":    raw["Muon_phi"],
    "mass":   raw["Muon_mass"],
    "charge": raw["Muon_charge"],
}, with_name="Momentum4D")

muons

In [ ]:
muon_plus  = muons[muons.charge > 0]
muon_minus = muons[muons.charge < 0]

mu1, mu2 = ak.unzip(ak.cartesian([muon_plus, muon_minus]))
dimuon_mass = ak.flatten((mu1 + mu2).mass)
dimuon_mass

In [ ]:
from hist import Hist

Hist.new.Reg(100, 0, 120, label="Dimuon mass (GeV)").Double().fill(dimuon_mass)

<br><br><br>

## Writing RNTuples

Since uproot v5.7, **assigning a dict to a file key creates an RNTuple by default**. This is the recommended way to write data.

### Quick write — assign a dict

In [ ]:
import numpy as np

# Skim: keep only events with at least 2 muons
with uproot.open("data/SMHiggsToZZTo4L.root:Events") as ntuple:
    dataset = ntuple.arrays(["nMuon", "Muon_pt", "Muon_eta", "Muon_charge"])

two_muon = dataset[dataset["nMuon"] == 2]

with uproot.recreate("/tmp/two_muon_skim.root") as f:
    # dict assignment → RNTuple (uproot >= 5.7)
    f["TwoMuon"] = {
        "nMuon":       np.asarray(two_muon["nMuon"]),
        "Muon_pt":     two_muon["Muon_pt"],
        "Muon_eta":    two_muon["Muon_eta"],
        "Muon_charge": two_muon["Muon_charge"],
    }

print("Written.")

In [ ]:
# Round-trip check
with uproot.open("/tmp/two_muon_skim.root:TwoMuon") as t:
    print(type(t).__name__, "—", t.num_entries, "events")
    print("Fields:", t.keys())

<br><br><br>

### `mkrntuple()` — explicit creation with a type spec

Use this when you need to initialize an empty RNTuple from a schema (e.g. to fill it in chunks):

In [ ]:
with uproot.recreate("/tmp/chunked.root") as f:
    # type spec: NumPy dtype string for flat fields, Awkward type string for ragged
    f.mkrntuple("Events", {
        "nMuon":   "int32",
        "Muon_pt": "var * float32",
    })
    # fill in chunks with .extend()
    for batch in uproot.iterate(
        "data/SMHiggsToZZTo4L.root:Events",
        expressions=["nMuon", "Muon_pt"],
        step_size=50_000,
    ):
        f["Events"].extend({
            "nMuon":   np.asarray(batch["nMuon"]),
            "Muon_pt": batch["Muon_pt"],
        })

with uproot.open("/tmp/chunked.root:Events") as t:
    print(t.num_entries, "events written in chunks")

<br><br><br>

### Writing a histogram

In [ ]:
counts, edges = np.histogram(np.asarray(ak.flatten(muons.pt)), bins=50, range=(0, 100))

with uproot.recreate("/tmp/muon_pt_hist.root") as f:
    f["h_muon_pt"] = (counts, edges)   # (values, edges) tuple → TH1

print("Histogram written.")

with uproot.open("/tmp/muon_pt_hist.root:h_muon_pt") as h:
    print(h)

<br><br><br>

## Remote access (HTTP and XRootD)

uproot reads ROOT files over HTTP or XRootD without downloading them first:

In [ ]:
with uproot.open("https://scikit-hep.org/uproot3/examples/Zmumu.root:events") as tree:
    print(tree.keys()[:6], "... (", tree.num_entries, "events)")

<br><br><br>

For CMS data on EOS, replace `http://` with `root://` (requires the `xrootd` Python package):

```python
uproot.open("root://eoscms.cern.ch//eos/cms/store/...file.root:Events")
```

<br><br><br>

## Reading legacy TTree files

The API for reading TTrees is identical to RNTuples — the same `.keys()`, `.show()`, `.array()`, `.arrays()`, `uproot.iterate()`, and `uproot.concatenate()` calls all work. The only differences are terminology (TTree says "branches"; RNTuple says "fields") and the class name you see when printing the object.

You may encounter TTree files from older CMS datasets (Run 2 NanoAOD and earlier). You can tell from the object's type:

In [ ]:
# Open a TTree file (older format)
with uproot.open("data/SMHiggsToZZTo4L.root") as f:
    for key, classname in f.classnames().items():
        print(f"{key:30s}  {classname}")

<br><br><br>

If the class name is `TTree`, you have a legacy file. Everything else in this notebook still applies — just replace "field" with "branch" when reading the uproot docs.

### Writing TTrees explicitly

If you need to write a TTree (e.g. for compatibility with older ROOT-based code), use `mktree()` instead of the default dict assignment:

In [ ]:
with uproot.recreate("/tmp/legacy_ttree.root") as f:
    # mktree() forces TTree creation regardless of uproot version
    f.mktree("Events", {
        "nMuon":   "int32",
        "Muon_pt": "var * float32",
    })
    f["Events"].extend({
        "nMuon":   np.asarray(two_muon["nMuon"]),
        "Muon_pt": two_muon["Muon_pt"],
    })

with uproot.open("/tmp/legacy_ttree.root:Events") as t:
    print(type(t).__name__, "—", t.num_entries, "events")

<br><br><br>

## Quick reference

| Task | Code |
|------|------|
| Open a file | `uproot.open("file.root")` |
| Jump to an object | `uproot.open("file.root:Name")` |
| List contents | `file.keys()` / `file.classnames()` |
| Inspect fields | `ntuple.show()` / `ntuple.keys()` |
| Count events | `ntuple.num_entries` |
| Read one field | `ntuple["field"].array()` |
| Read several fields | `ntuple.arrays(["f1", "f2"])` |
| Read by glob | `ntuple.arrays(filter_name="Muon_*")` |
| Read by regex | `ntuple.arrays(filter_name=re.compile(r"..."))` |
| Read a slice | `ntuple.arrays(entry_start=0, entry_stop=1000)` |
| Choose output type | `library="ak"` (default) / `"np"` / `"pd"` |
| Chunked iteration | `uproot.iterate("file.root:T", step_size=10_000)` |
| Many files | `uproot.iterate(["a.root:T", "b.root:T"])` or glob `"*.root:T"` |
| All → one array | `uproot.concatenate("*.root:T")` |
| Write new file | `uproot.recreate("out.root")` |
| Write RNTuple (default) | `f["Name"] = {"field": array}` |
| Write RNTuple (explicit) | `f.mkrntuple("Name", {"field": "type"})` then `.extend()` |
| Write TTree (legacy) | `f.mktree("Name", {"branch": "type"})` then `.extend()` |
| Write histogram | `f["h"] = (counts, edges)` |


<br><br><br>

## Puzzles

<br><br><br>

### Puzzle 1

Open `data/SMHiggsToZZTo4L.root` and print the number of events in the `Events` object **without reading any field data**.

<br><br><br>

#### Solution

In [ ]:
with uproot.open("data/SMHiggsToZZTo4L.root:Events") as ntuple:
    print(ntuple.num_entries)

<br><br><br>

### Puzzle 2

Read only `Electron_pt` and `Electron_eta` for the first 5,000 events. Print the mean electron $p_T$ across all electrons in those events.

<br><br><br>

#### Solution

In [ ]:
with uproot.open("data/SMHiggsToZZTo4L.root:Events") as ntuple:
    e = ntuple.arrays(["Electron_pt", "Electron_eta"], entry_stop=5000)

print(f"Mean electron pT: {ak.mean(ak.flatten(e['Electron_pt'])):.2f} GeV")

<br><br><br>

### Puzzle 3

Read all fields whose names end in `_pt` using a regular expression. How many such fields are there, and what are their names?

<br><br><br>

#### Solution

In [ ]:
import re

with uproot.open("data/SMHiggsToZZTo4L.root:Events") as ntuple:
    pt_fields = ntuple.keys(filter_name=re.compile(r".+_pt$"))

print(f"{len(pt_fields)} fields ending in _pt:")
for name in pt_fields:
    print(" ", name)

<br><br><br>

### Puzzle 4

Use `uproot.iterate()` with `step_size=50_000` to count the total number of electrons across **all** events in `data/SMHiggsToZZTo4L.root`.

<br><br><br>

#### Solution

In [ ]:
total_electrons = 0

for batch in uproot.iterate(
    "data/SMHiggsToZZTo4L.root:Events",
    expressions=["nElectron"],
    step_size=50_000,
):
    total_electrons += int(ak.sum(batch["nElectron"]))

print(f"Total electrons: {total_electrons:,}")

<br><br><br>

### Puzzle 5

Write a skim: keep only events with `nElectron >= 2` **and** `nMuon >= 2`, save the fields `Electron_pt`, `Electron_charge`, `Muon_pt`, `Muon_charge` to `/tmp/four_lepton_skim.root` as an RNTuple named `FourLepton`. Verify by reopening the file and printing the event count and field names.

<br><br><br>

#### Solution

In [ ]:
with uproot.open("data/SMHiggsToZZTo4L.root:Events") as ntuple:
    dataset = ntuple.arrays(["nElectron", "nMuon",
                             "Electron_pt", "Electron_charge",
                             "Muon_pt", "Muon_charge"])

cut = (dataset["nElectron"] >= 2) & (dataset["nMuon"] >= 2)
skimmed = dataset[cut]

with uproot.recreate("/tmp/four_lepton_skim.root") as f:
    f["FourLepton"] = {
        "Electron_pt":     skimmed["Electron_pt"],
        "Electron_charge": skimmed["Electron_charge"],
        "Muon_pt":         skimmed["Muon_pt"],
        "Muon_charge":     skimmed["Muon_charge"],
    }

with uproot.open("/tmp/four_lepton_skim.root:FourLepton") as t:
    print(type(t).__name__, "—", t.num_entries, "four-lepton events")
    print("Fields:", t.keys())

<br><br><br>